In [15]:
import numpy as np
import matplotlib.pyplot as plt
import random
import math

class Value:

    # a Value represents a single scalar value and its gradient
    def __init__(self, data, _children=()):
        self.data = data
        self.grad = 0
        self._backward = lambda: None
        self._prev = set(_children)

    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward

        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out
    
    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data**other, (self,))

        def _backward():
            self.grad += (other * self.data**(other-1)) * out.grad
        out._backward = _backward

        return out
    
    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self, ))

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward

        return out
    
    def log(self):
        x = self.data
        out = Value(math.log(x), (self, ))

        def _backward():
            self.grad += (1 / x) * out.grad
        out._backward = _backward

        return out
    
    def sigmoid(self):
        x = self.data
        out = Value(1 / (1 + math.exp(-x)), (self, ))

        def _backward():
            s = out.data
            self.grad += (s * (1 - s)) * out.grad
        out._backward = _backward

        return out
    
    def backward(self):

        # topological order all of the children in the graph
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        # go one variable at a time and apply the chain rule to get its gradient
        self.grad = 1
        for v in reversed(topo):
            v._backward()

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"
    
    def __neg__(self): # -self
        return self * -1
    
    def __sub__(self, other): # self - other
        return self + (-other)
    
    def __rsub__(self, other): # other - self
        return other + (-self)
    
    def __radd__(self, other): # other + self
        return self + other
    
    def __rmul__(self, other): # other * self
        return self * other

In [19]:
x1 = Value(2.0)
w1 = Value(-3.0)

x2 = Value(1.0)
w2 = Value(1.0)

x3 = Value(-1.0)
w3 = Value(2.0)

b = Value(1.0)

y = Value(1.0)  # target

summed = x1*w1 + x2*w2 + x3*w3 + b
yhat = summed.sigmoid()

loss = -(y*(yhat.log()) + (1 - y)*(1 - yhat).log())
loss.backward()
loss

Value(data=6.00247568513773, grad=1)

In [20]:
x1, w1, x2, w2, x3, w3, b

(Value(data=2.0, grad=2.9925821305300957),
 Value(data=-3.0, grad=-1.9950547536867305),
 Value(data=1.0, grad=-0.9975273768433652),
 Value(data=1.0, grad=-0.9975273768433652),
 Value(data=-1.0, grad=-1.9950547536867305),
 Value(data=2.0, grad=0.9975273768433652),
 Value(data=1.0, grad=-0.9975273768433652))

In [ ]:
X = np.array([
    [1, 2],
    [2, 3],
    [3, 4],
    [4, 5]
])
y = np.array([0, 0, 1, 1])
n_features = X.shape[1]
w = [Value(random.uniform(-1, 1)) for _ in range(n_features)]
b = Value(0.0)
sum(wi * xi for wi, xi in zip(w, X[0])) + b